# Data Catalog Configuration for Amazon SageMaker Feature Store

This notebook demonstrates all available `DataCatalogConfig` options when creating a Feature Group (FG) with an offline store. The `DataCatalogConfig` controls how your Feature Store offline data is registered in the AWS Glue Data Catalog — including the table name, database name, and catalog name.

## Overview

When you create a Feature Group with an offline store, Feature Store creates an AWS Glue table (or Apache Iceberg table) to catalog your data. You can either:

1. **Let Feature Store auto-generate** table names (default behavior)
2. **Specify custom names** for the table and database
3. **Bring Your Own Table (BYOT)** — point to an existing Glue table you manage yourself

This notebook covers all scenarios with working examples and explains common pitfalls.

## Configuration Reference

| DisableGlueTableCreation | DataCatalogConfig | TableFormat | Behavior | Schema Evolution | Notes |
|---|---|---|---|---|---|
| `False` (default) | Not provided | Glue/Iceberg | Auto-generate names | Automatic | Simplest option. Table name derived from Feature Group name. |
| `False` | Provided | Glue | Create Glue table with custom names | Automatic | Names must be Athena-compatible (lowercase, underscores only). If table already exists, FG goes to CreateFailed. |
| `False` | Provided | Iceberg | Create Iceberg table with custom names | Automatic | Same naming rules. Database created if it doesn't exist. |
| `True` | Provided | Glue | Associate table (BYOT) | **Manual** — you must maintain the table | For BYOT, it is the user's responsibility to maintain the table for its entire lifespan. |
| `True` | Not provided | Glue | No table created or associated | — | Data written to S3 but not queryable via Athena without manual table creation. |
| `True` | Any | Iceberg | Error | — | Iceberg requires Feature Store to manage the table. BYOT not supported. |

**Important rules:**
- **Catalog**: Must always be `AwsDataCatalog` — no other catalog names are supported
- **Naming**: Table and database names must be Athena-compatible — lowercase alphanumeric and underscores only, no hyphens or spaces
- **Uniqueness**: When `DisableGlueTableCreation=False` with custom names, the table must not already exist (use BYOT if it does)
- **Schema updates**: Feature Store auto-updates the Glue/Iceberg table schema only when it created the table. For BYOT, you must maintain and update the table for its entire lifespan.

## Prerequisites

- An AWS account with SageMaker Feature Store access
- IAM role with `AmazonSageMakerFeatureStoreAccess` and `AmazonSageMakerFullAccess` policies
- An S3 bucket for offline store data
- `AWS_DEFAULT_REGION` set if not passing a region in function calls

In [ ]:
# Install/upgrade SageMaker SDK if needed
# !pip install --upgrade sagemaker

import boto3
import sagemaker
import pandas as pd
import time
import uuid
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.feature_store.feature_definition import FeatureDefinition, FeatureTypeEnum
from sagemaker.feature_store.inputs import DataCatalogConfig, TableFormatEnum

# Session setup
boto_session = boto3.Session()
region = boto_session.region_name
sagemaker_session = Session(boto_session=boto_session)
role = sagemaker.get_execution_role()

# S3 bucket for offline store
default_bucket = sagemaker_session.default_bucket()
offline_store_s3_uri = f"s3://{default_bucket}/feature-store-data-catalog-config-demo"

print(f"Region: {region}")
print(f"Role: {role}")
print(f"Offline Store S3 URI: {offline_store_s3_uri}")

In [ ]:
# Create sample data for demonstrations
customer_data = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004", "C005"],
    "age": [25, 30, 35, 40, 45],
    "city": ["Seattle", "Portland", "Denver", "Austin", "Chicago"],
    "event_time": [
        "2026-01-01T00:00:00Z",
        "2026-01-01T00:00:00Z",
        "2026-01-01T00:00:00Z",
        "2026-01-01T00:00:00Z",
        "2026-01-01T00:00:00Z",
    ],
})

# Feature definitions
feature_definitions = [
    FeatureDefinition(feature_name="customer_id", feature_type=FeatureTypeEnum.STRING),
    FeatureDefinition(feature_name="age", feature_type=FeatureTypeEnum.INTEGRAL),
    FeatureDefinition(feature_name="city", feature_type=FeatureTypeEnum.STRING),
    FeatureDefinition(feature_name="event_time", feature_type=FeatureTypeEnum.STRING),
]

print("Sample data:")
customer_data

In [ ]:
def create_feature_group(name_suffix, create_kwargs, expect_error=False):
    """Create a feature group and wait for it to become active.

    Args:
        name_suffix: suffix for the feature group name
        create_kwargs: dict of keyword arguments for FeatureGroup.create()
        expect_error: if True, expect the call to raise an exception
    Returns:
        FeatureGroup object on success, None on expected error
    """
    fg_name = f"data-catalog-demo-{name_suffix}-{uuid.uuid4().hex[:6]}"

    fg = FeatureGroup(name=fg_name, sagemaker_session=sagemaker_session)
    fg.feature_definitions = feature_definitions

    try:
        fg.create(
            record_identifier_name="customer_id",
            event_time_feature_name="event_time",
            role_arn=role,
            enable_online_store=False,
            **create_kwargs,
        )

        if expect_error:
            print(f"Expected an error but creation succeeded for: {fg_name}")
            get_data_catalog_config(fg)
        else:
            print(f"Feature Group created: {fg_name}")
            status = "Creating"
            while status == "Creating":
                time.sleep(5)
                status = fg.describe()["FeatureGroupStatus"]
            print(f"   Status: {status}")

        return fg

    except Exception as e:
        if expect_error:
            print(f"Expected error received: {type(e).__name__}")
            print(f"   Message: {str(e)[:300]}")
        else:
            print(f"Unexpected error: {type(e).__name__}")
            print(f"   Message: {str(e)[:300]}")
        return None


def get_data_catalog_config(fg):
    """Fetches DataCatalogConfig from DescribeFeatureGroup API response."""
    if fg is None:
        return
    desc = fg.describe()
    offline_config = desc.get("OfflineStoreConfig", {})
    catalog_config = offline_config.get("DataCatalogConfig")
    table_format = offline_config.get("TableFormat", "Glue")

    print(f"\nDescribeFeatureGroup API response:")
    print(f"   FeatureGroupName: {desc['FeatureGroupName']}")
    print(f"   TableFormat: {table_format}")
    if catalog_config:
        print(f"   DataCatalogConfig:")
        print(f"     TableName: {catalog_config['TableName']}")
        print(f"     Database:  {catalog_config['Database']}")
        print(f"     Catalog:   {catalog_config['Catalog']}")
    else:
        print(f"   DataCatalogConfig: None (no table associated)")


# Track feature groups for cleanup
feature_groups_to_cleanup = []

## Scenario 1: Default Behavior — Auto-Generated Names (Glue Table Format)

When you create a Feature Group without specifying `DataCatalogConfig`, Feature Store automatically:
- Generates a table name based on the Feature Group name (sanitized for Athena compatibility)
- Uses the default database: `sagemaker_featurestore`
- Uses the default catalog: `AwsDataCatalog`

This is the simplest configuration and works for most use cases.

In [ ]:
# Scenario 1: Auto-generated names with Glue table format (default)
# Note: disable_glue_table_creation defaults to False, so omitting it
# is the same as explicitly passing disable_glue_table_creation=False
fg1 = create_feature_group("auto-glue", {
    "s3_uri": offline_store_s3_uri,
})

# Fetch DataCatalogConfig from DescribeFeatureGroup API
get_data_catalog_config(fg1)

if fg1:
    feature_groups_to_cleanup.append(fg1)

## Scenario 2: Default Behavior — Auto-Generated Names (Iceberg Table Format)

Same as Scenario 1, but using the Apache Iceberg table format. Iceberg is recommended for streaming workloads because it supports efficient compaction of small files.

Note: For Iceberg format, `disable_glue_table_creation` must be `False` (the default).

In [ ]:
# Scenario 2: Auto-generated names with Iceberg table format
fg2 = create_feature_group("auto-iceberg", {
    "s3_uri": offline_store_s3_uri,
    "table_format": TableFormatEnum.ICEBERG,
})

# Fetch DataCatalogConfig from DescribeFeatureGroup API
get_data_catalog_config(fg2)

if fg2:
    feature_groups_to_cleanup.append(fg2)

## Scenario 3: Custom Glue Table Names

You can specify your own table name and database name using `DataCatalogConfig`. This is useful when:
- You want human-readable table names (instead of auto-generated ones)
- You want to organize Feature Groups into specific databases
- You're integrating with existing data lake naming conventions

**Rules for custom names:**
- Names must be Athena-compatible: lowercase alphanumeric and underscores only
- Must start with a letter or underscore
- Table names: max 252 characters
- Database names: max 255 characters
- Catalog must be `AwsDataCatalog` (only supported option)
- The table must NOT already exist (Feature Store creates it for you)

In [ ]:
# Scenario 3: Custom Glue table names
custom_table = f"customer_features_prod_{uuid.uuid4().hex[:6]}"
custom_database = f"my_ml_features_{uuid.uuid4().hex[:6]}"

print(f"Creating with custom names:")
print(f"  Table: {custom_table}")
print(f"  Database: {custom_database}")

fg3 = create_feature_group("custom-glue", {
    "s3_uri": offline_store_s3_uri,
    "disable_glue_table_creation": False,
    "data_catalog_config": DataCatalogConfig(
        table_name=custom_table,
        database=custom_database,
        catalog="AwsDataCatalog",
    ),
})

# Fetch DataCatalogConfig from DescribeFeatureGroup API
get_data_catalog_config(fg3)

if fg3:
    feature_groups_to_cleanup.append(fg3)

## Scenario 4: Custom Iceberg Table Names

The same custom naming works for Iceberg tables. Feature Store creates the Iceberg table in the specified database with your chosen table name.

Feature Store also creates the database if it doesn't already exist.

In [ ]:
# Scenario 4: Custom Iceberg table names
custom_iceberg_table = f"customer_features_iceberg_{uuid.uuid4().hex[:6]}"
custom_iceberg_database = f"ml_iceberg_store_{uuid.uuid4().hex[:6]}"

print(f"Creating Iceberg table with custom names:")
print(f"  Table: {custom_iceberg_table}")
print(f"  Database: {custom_iceberg_database}")

fg4 = create_feature_group("custom-iceberg", {
    "s3_uri": offline_store_s3_uri,
    "disable_glue_table_creation": False,
    "table_format": TableFormatEnum.ICEBERG,
    "data_catalog_config": DataCatalogConfig(
        table_name=custom_iceberg_table,
        database=custom_iceberg_database,
        catalog="AwsDataCatalog",
    ),
})

# Fetch DataCatalogConfig from DescribeFeatureGroup API
get_data_catalog_config(fg4)

if fg4:
    feature_groups_to_cleanup.append(fg4)

## Scenario 5: Bring Your Own Table (BYOT) — Glue Format

If you already have an existing Glue table and want Feature Store to write data to it, use `disable_glue_table_creation=True` with `data_catalog_config` pointing to your existing table.

**Important considerations for BYOT mode:**
- The table must already exist before creating the Feature Group
- Table schema management is the user's responsibility
- If you later add features via `UpdateFeatureGroup`, you must **manually** update the Glue table schema (e.g., using `ALTER TABLE ADD COLUMNS`)
- Feature Store only automatically updates the schema when it created the table itself
- Your table must have the correct schema matching your feature definitions (including system columns: `write_time`, `api_invocation_time`, `is_deleted`)
- The table's S3 location must point to the same path as your offline store S3 URI
- The data format must be Parquet for Athena queries to work correctly
- The table must use `EXTERNAL_TABLE` type

**Note:** BYOT is only supported for Glue table format. Iceberg tables must always be created by Feature Store.

In [ ]:
# Scenario 5: BYOT — First, create a Glue table manually
glue_client = boto3.client("glue")
byot_database = f"byot_demo_db_{uuid.uuid4().hex[:6]}"
byot_table_name = f"byot_customer_features_{uuid.uuid4().hex[:6]}"

# Create database
try:
    glue_client.create_database(
        DatabaseInput={"Name": byot_database, "Description": "Demo database for BYOT"}
    )
    print(f"Database created: {byot_database}")
except glue_client.exceptions.AlreadyExistsException:
    print(f"   Database already exists: {byot_database}")

# Create a Glue table manually
glue_client.create_table(
    DatabaseName=byot_database,
    TableInput={
        "Name": byot_table_name,
        "StorageDescriptor": {
            "Columns": [
                {"Name": "customer_id", "Type": "string"},
                {"Name": "age", "Type": "bigint"},
                {"Name": "city", "Type": "string"},
                {"Name": "event_time", "Type": "string"},
                {"Name": "write_time", "Type": "timestamp"},
                {"Name": "api_invocation_time", "Type": "timestamp"},
                {"Name": "is_deleted", "Type": "boolean"},
            ],
            "Location": f"{offline_store_s3_uri}/byot-demo/",
            "InputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat",
            "OutputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat",
            "SerdeInfo": {
                "SerializationLibrary": "org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe"
            },
        },
        "TableType": "EXTERNAL_TABLE",
    },
)
print(f"Glue table created: {byot_database}.{byot_table_name}")

# Now create Feature Group pointing to this existing table
print(f"\nCreating Feature Group with BYOT pointing to existing table...")
fg5 = create_feature_group("byot-glue", {
    "s3_uri": offline_store_s3_uri,
    "disable_glue_table_creation": True,
    "data_catalog_config": DataCatalogConfig(
        table_name=byot_table_name,
        database=byot_database,
        catalog="AwsDataCatalog",
    ),
})

# Fetch DataCatalogConfig from DescribeFeatureGroup API
get_data_catalog_config(fg5)

if fg5:
    feature_groups_to_cleanup.append(fg5)

## Scenario 6: DisableGlueTableCreation=True without DataCatalogConfig

When you set `disable_glue_table_creation=True` and do NOT provide `data_catalog_config`, Feature Store writes data to S3 but does NOT create or associate any Glue table. No table metadata is stored with the feature group.

You can't query the data via Athena directly — you'd need to create a Glue table yourself afterward pointing to the S3 location.

In [ ]:
# Scenario 6: DisableGlueTableCreation=True, no DataCatalogConfig
fg6 = create_feature_group("no-table", {
    "s3_uri": offline_store_s3_uri,
    "disable_glue_table_creation": True,
})

# Fetch DataCatalogConfig from DescribeFeatureGroup API
get_data_catalog_config(fg6)

if fg6:
    feature_groups_to_cleanup.append(fg6)

## Querying Custom-Named Tables with Amazon Athena

One of the main benefits of custom `DataCatalogConfig` is that your tables have human-readable names in Athena. Let's ingest some data and then query the custom-named table to prove it works end-to-end.

Note: Data takes a few minutes to replicate from Feature Store to the offline store before it's queryable via Athena.

In [ ]:
# Ingest data into the custom Glue table feature group (fg3)
if fg3:
    print("Ingesting records into feature group with custom Glue table name...")

    from sagemaker.feature_store.inputs import FeatureValue

    for _, row in customer_data.iterrows():
        record = [
            FeatureValue(feature_name="customer_id", value_as_string=str(row["customer_id"])),
            FeatureValue(feature_name="age", value_as_string=str(row["age"])),
            FeatureValue(feature_name="city", value_as_string=str(row["city"])),
            FeatureValue(feature_name="event_time", value_as_string=str(row["event_time"])),
        ]
        fg3.put_record(record)

    print(f"Ingested {len(customer_data)} records")
    print("   Waiting for data to replicate to offline store (this may take a few minutes)...")

In [ ]:
# Wait for data to appear in the offline store
if fg3:
    max_wait_minutes = 10
    poll_interval_seconds = 30
    elapsed = 0
    data_available = False

    query = fg3.athena_query()
    print(f"Table name in Athena: {query.database}.{query.table_name}")
    print(f"(Expected: {custom_database}.{custom_table})\n")

    while elapsed < max_wait_minutes * 60:
        try:
            query.run(
                query_string=f'SELECT COUNT(*) as cnt FROM "{query.table_name}"',
                output_location=f"s3://{default_bucket}/athena-query-results/",
            )
            query.wait()
            df_count = query.as_dataframe()
            count = int(df_count["cnt"].iloc[0])

            if count > 0:
                print(f"Data available! {count} records found in offline store.")
                data_available = True
                break
        except Exception:
            pass

        elapsed += poll_interval_seconds
        print(f"   Waiting... ({elapsed}s elapsed)")
        time.sleep(poll_interval_seconds)

    if not data_available:
        print(f"Data not available after {max_wait_minutes} minutes. "
              "You can re-run this cell later.")

In [ ]:
# Query the custom-named table via Athena
if fg3 and data_available:
    query = fg3.athena_query()

    query_string = f"""
    SELECT customer_id, age, city, event_time
    FROM "{query.table_name}"
    ORDER BY age
    LIMIT 10
    """

    print(f"Running Athena query on: {query.database}.{query.table_name}")
    print(f"Query: {query_string.strip()}\n")

    query.run(
        query_string=query_string,
        output_location=f"s3://{default_bucket}/athena-query-results/",
    )
    query.wait()

    df_results = query.as_dataframe()
    print("Query results:")
    print(df_results.to_string(index=False))
    print(f"\n   Custom table '{query.table_name}' in database '{query.database}' is fully queryable!")

## Error Scenarios — What NOT to Do

The following scenarios demonstrate common mistakes that result in errors. Understanding these helps avoid confusion during feature group creation.

### Error: Custom Catalog Name (Not AwsDataCatalog)

Only `AwsDataCatalog` is supported as the catalog name. Any other value results in a validation error.

In [ ]:
# Error: Custom catalog name -> ValidationError
create_feature_group("error-catalog", {
    "s3_uri": offline_store_s3_uri,
    "disable_glue_table_creation": False,
    "data_catalog_config": DataCatalogConfig(
        table_name="my_table",
        database="my_database",
        catalog="MyCatalog",  # Only "AwsDataCatalog" is supported
    ),
}, expect_error=True)

### Error: Table Already Exists (with DisableGlueTableCreation=False)

When `disable_glue_table_creation=False` and you provide a `data_catalog_config`, Feature Store attempts to **create** the table. If a table with that name already exists in the specified database, the `CreateFeatureGroup` API call still returns a Feature Group ARN, but the Feature Group will NOT reach Active status — it will transition to `CreateFailed`.

If you want to attach to an existing table, use `disable_glue_table_creation=True` instead (BYOT mode).

In [ ]:
# Table already exists — FG creates but goes to CreateFailed
fg_exists = create_feature_group("error-exists", {
    "s3_uri": offline_store_s3_uri,
    "disable_glue_table_creation": False,
    "data_catalog_config": DataCatalogConfig(
        table_name=byot_table_name,  # This table already exists from Scenario 5!
        database=byot_database,
        catalog="AwsDataCatalog",
    ),
})

# Poll status — the FG will NOT reach Active, it will go to CreateFailed
if fg_exists:
    print("\nPolling Feature Group status...")
    for _ in range(12):
        time.sleep(10)
        desc = fg_exists.describe()
        status = desc["FeatureGroupStatus"]
        print(f"   Status: {status}")
        if status != "Creating":
            break
    if status == "CreateFailed":
        failure_reason = desc.get("FailureReason", "Unknown")
        print(f"\n   FailureReason: {failure_reason}")

    feature_groups_to_cleanup.append(fg_exists)

### Error: Invalid Table/Database Names

Table and database names must be Athena-compatible:
- Lowercase alphanumeric characters and underscores only
- Must start with a letter or underscore
- No hyphens, spaces, or special characters

In [ ]:
# Error: Hyphenated table name -> ValidationError
create_feature_group("error-hyphen", {
    "s3_uri": offline_store_s3_uri,
    "disable_glue_table_creation": False,
    "data_catalog_config": DataCatalogConfig(
        table_name="my-feature-table",  # Hyphens not allowed
        database="my_database",
        catalog="AwsDataCatalog",
    ),
}, expect_error=True)

### Error: BYOT with Non-Existent Table

When using `disable_glue_table_creation=True` with `data_catalog_config`, it is the user's responsibility to ensure the table exists and is correctly configured.

The `CreateFeatureGroup` call will succeed and the Feature Group will reach Active status. However, Athena queries will fail because the table doesn't actually exist in the Glue Data Catalog.

In [ ]:
# BYOT pointing to a table that doesn't exist — FG still creates successfully
fg_no_table = create_feature_group("error-no-table", {
    "s3_uri": offline_store_s3_uri,
    "disable_glue_table_creation": True,
    "data_catalog_config": DataCatalogConfig(
        table_name="this_table_does_not_exist",
        database="this_database_does_not_exist",
        catalog="AwsDataCatalog",
    ),
})

# FG reaches Active status — but the table doesn't exist!
if fg_no_table:
    # Fetch DataCatalogConfig from DescribeFeatureGroup API
    get_data_catalog_config(fg_no_table)

    # Attempting to query will fail because the table doesn't exist
    print("\nAttempting Athena query on non-existent table...")
    try:
        query = fg_no_table.athena_query()
        query.run(
            query_string=f'SELECT * FROM "{query.table_name}" LIMIT 1',
            output_location=f"s3://{default_bucket}/athena-query-results/",
        )
        query.wait()
        # Check if query actually succeeded
        df = query.as_dataframe()
        if df.empty:
            print("   Query returned no results (table does not exist).")
        else:
            print("   Query unexpectedly succeeded.")
    except Exception as e:
        print(f"   Query failed as expected: {type(e).__name__}")
        print(f"   Message: {str(e)[:200]}")

    print("\n   This confirms: for BYOT, the user is responsible for ensuring the table exists.")

    feature_groups_to_cleanup.append(fg_no_table)

### Error: Iceberg with DisableGlueTableCreation=True

Iceberg tables must be created and managed by Feature Store. The BYOT pattern (`disable_glue_table_creation=True`) is NOT supported for Iceberg format because Feature Store needs to actively manage the Iceberg table metadata during data replication.

In [ ]:
# Error: Iceberg + disable_glue_table_creation=True -> Not supported
create_feature_group("error-iceberg-byot", {
    "s3_uri": offline_store_s3_uri,
    "disable_glue_table_creation": True,
    "table_format": TableFormatEnum.ICEBERG,
    "data_catalog_config": DataCatalogConfig(
        table_name="my_iceberg_table",
        database="my_database",
        catalog="AwsDataCatalog",
    ),
}, expect_error=True)

## Summary

In this notebook, we covered all `DataCatalogConfig` scenarios — refer to the **Configuration Reference** table at the top for the complete behavior matrix.

### Further Reading
- [OfflineStoreConfig API Reference](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_OfflineStoreConfig.html)
- [DataCatalogConfig API Reference](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_DataCatalogConfig.html)
- [Athena Naming Conventions](https://docs.aws.amazon.com/athena/latest/ug/tables-databases-columns-names.html)
- [Iceberg Metadata Management](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store-iceberg-metadata-management.html)

In [ ]:
# Cleanup: Delete all Feature Groups created in this notebook
print("Cleaning up Feature Groups...\n")

for fg in feature_groups_to_cleanup:
    try:
        fg.delete()
        print(f"Deleted: {fg.name}")
    except Exception as e:
        print(f"Could not delete {fg.name}: {e}")

# Clean up the manually created Glue table and database
try:
    glue_client.delete_table(DatabaseName=byot_database, Name=byot_table_name)
    print(f"\nDeleted Glue table: {byot_database}.{byot_table_name}")
except Exception as e:
    print(f"Could not delete table: {e}")

try:
    glue_client.delete_database(Name=byot_database)
    print(f"Deleted database: {byot_database}")
except Exception as e:
    print(f"Could not delete database: {e}")

print("\nDone! All resources cleaned up.")